# Upload dataset on Google Cloud

This jupyter notebook take the first thousand samples of pixelart dataset on hugging face and uploads it to a google cloud storage bucket.

In [ ]:
import os
import shutil

import PIL
import pandas as pd
from google.cloud import storage

In [20]:
from datasets import load_dataset, Image

ds = load_dataset("jainr3/diffusiondb-pixelart", "2k_first_1k")

In [12]:
ds

DatasetDict({
    train: Dataset({
        features: ['image', 'text'],
        num_rows: 1000
    })
})

In [ ]:
os.makedirs("dataset", exist_ok=True)
labels = pd.DataFrame(columns = ["text", "image_path"])
for i, example in enumerate(ds["train"]):
    # Save each PIL image in the dataset to the dataset folder
    img = example["image"]
    img.save(f"dataset/image_{i + 1}.png")
    # Save the text label in the labels dataframe
    labels.loc[i, "text"] = example["text"]
    labels.loc[i, "image_path"] = f"dataset/image_{i + 1}.png"

In [10]:
# Save the labels to a CSV file
labels.to_csv("dataset/labels.csv", index=False)

## Uploading the dataset to Cloud Storage

In [ ]:
# Zip the dataset folder (images + labels.csv) and upload it for fine_tune.ipynb
shutil.make_archive("dataset", "zip", root_dir=".", base_dir="dataset")
bucket = storage.Client().bucket("pixel-art-dataset")
bucket.blob("dataset.zip").upload_from_filename("dataset.zip")